In [ ]:
from google.colab import drive
drive.mount('/content/drive')
#Important - this code requires connecting to Google Drive.
#Before executing the code, create a folder on Google Drive called "datasets" and load .dta files there
#It is better to use datasets from 2020 and later for this code; otherwise the code may error out due to missing data. In that case, use WBES_analysis_for_incomplete_extended.ipynb

In [ ]:
#PLEASE READ THE INSTRUCTIONS BELOW:
#If you wish to omit manual validation process and proceed with pre-picked firms (they are same as in thesis analysis), please do the following:
#1. In Colab in the files tab import approved_ids.txt (from ZIP file)
#2. When at the bootom of the page, type YY and Enter
#
#If you wish to validate the data by yourself, please do the following:
#1. strict_ict shows firms that are selected automatically by ISIC code matching; Candidates should be selected manually from the table.
#2. When at the bootom of the page, type Y and Enter. If the amount of candidates is 0, simply press Enter to proceed. If candidate table is shown, write down their idstd to choose candidates from the list, seperating them by a comma and press Enter to proceed.
#
#After running this cell, please scroll down for data validation process
#You can download the results by clicking the results folder in the folder tab on the left and getting master_ict_dataset.xlsx. Alternatively, you may import it to drive by excecuting the next cell.
import os
import logging
import time
from typing import Optional

import numpy as np
import pandas as pd
!pip install pyreadstat
import pyreadstat

from pathlib import Path
import re
import os
import logging

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter
from google.colab import data_table


# ── Logging ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger(__name__)


# ── Constants ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# WBES negative sentinel codes representing non-response / not applicable
WBES_MISSING_CODES: list = [-9, -8, -7, -6, -5, -4, -3, -2, -1]

# Comprehensive ICT classification based on ISIC Rev. 4 (4-digit codes)
ICT_ISIC_CLASSES: list = [
    # Division 58 – Publishing activities
    5811, 5812, 5813, 5819, 5820,
    # Division 59 – Motion picture, video and TV, music publishing
    5911, 5912, 5913, 5914, 5920,
    # Division 60 – Programming and broadcasting activities
    6010, 6020,
    # Division 61 – Telecommunications
    6110, 6120, 6130, 6190,
    # Division 62 – Computer programming, consultancy and related
    6201, 6202, 6209, 6203,
    # Division 63 – Information service activities
    6311, 6312, 6391, 6399
]

# Variables always included; extend via the additional_vars parameter
BASE_VARS: list = ["idstd", "b2b", "a4a", "d1a2_v4", "strata",
    "panel"]

# Variables whose raw numeric values must NOT be replaced with string labels
NUMERIC_PRESERVE: set = {"b2b", "d1a2_v4"}

# Column preference order for the free-text activity description field
ACTIVITY_DESC_COLS: list = ["d1a1x", "a4a"]

# Default keyword list for Stage 1 candidate screening
DEFAULT_ICT_KEYWORDS: list = [
    "software", "telecom", "telecommunications", "internet",
    "information technology", "information systems",
    "computer", "computing", "network", "networking",
    "hosting", "cloud", "cybersecurity", "cyber security",
    "data center", "datacenter", "mobile communications",
    "ict", "digital services", "it services",
]


# ── Private helpers ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _replace_wbes_missing(series: pd.Series, missing_codes: list) -> pd.Series:
    """Replace WBES negative sentinel integers with np.nan."""
    if pd.api.types.is_numeric_dtype(series):
        # Explicitly infer objects to retain old behavior as suggested by FutureWarning
        return series.replace(missing_codes, np.nan).infer_objects(copy=False)
    str_codes = [str(c) for c in missing_codes] + [f"{c}.0" for c in missing_codes]
    # Explicitly infer objects to retain old behavior as suggested by FutureWarning
    return series.replace(str_codes, np.nan).infer_objects(copy=False)


def _apply_value_labels(
    df: pd.DataFrame,
    meta: "pyreadstat.metadata_container",
    skip_cols: set,
) -> pd.DataFrame:
    """Map raw integer codes to human-readable Stata value labels."""
    label_map: dict = meta.variable_value_labels

    for col in df.columns:
        if col in skip_cols or col not in label_map or not label_map[col]:
            continue

        try:
            col_map = {float(k): v for k, v in label_map[col].items()}
        except (TypeError, ValueError):
            col_map = label_map[col]

        nan_mask = df[col].isna()
        mapped = df[col].map(col_map)

        df[col] = mapped.where(mapped.notna() | nan_mask, other=df[col])
        df[col] = df[col].where(~nan_mask, other=np.nan)

    return df


def _build_ict_masks(
    df: pd.DataFrame,
    approved_extra_ids: list,
    isic_classes: list = ICT_ISIC_CLASSES,
) -> tuple:
    """Return (ict_strict, ict_extended) boolean Series based on 4-digit codes."""
    # Use full 4-digit code as requested
    full_isic = pd.to_numeric(df["d1a2_v4"], errors="coerce")
    ict_strict   = full_isic.isin(isic_classes)
    ict_manual   = df["idstd"].isin(approved_extra_ids)
    ict_extended = ict_strict | ict_manual
    return ict_strict, ict_extended


def _match_keywords(text: str, keywords: list) -> str:
    """Return a comma-separated string of keywords found in text, or ''. """
    if pd.isna(text):
        return ""
    text_lower = str(text).lower()
    return ", ".join(k for k in keywords if k in text_lower)

def discover_datasets(dataset_folder: str) -> list:
    folder = Path(dataset_folder)
    if not folder.exists():
        raise FileNotFoundError(f"Dataset folder not found: {dataset_folder}")
    datasets = sorted(folder.glob("*.dta"))
    if not datasets:
        raise FileNotFoundError(f"No .dta files found in {dataset_folder}")
    return [str(f) for f in datasets]


def parse_dataset_metadata(file_path: str) -> dict:
    filename = Path(file_path).name
    match = re.match(r"^(.*?)\-(\d{4})", filename)
    if match:
        country = match.group(1)
        year = int(match.group(2))
    else:
        country = "Unknown"
        year = np.nan
    return {"country": country, "year": year, "source_file": filename}

def detect_weight_variable(df: pd.DataFrame) -> Optional[str]:
    priority = ["wmedian", "wstrong", "wweak"]
    for var in priority:
        if var in df.columns:
            return var
    return None

def load_approval_database(approval_file="approved_ids.txt"):
    approvals = {}
    if not os.path.exists(approval_file):
        return approvals
    current_key = None
    with open(approval_file, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            if line.startswith("[") and line.endswith("]"):
                current_key = line[1:-1]
                approvals[current_key] = []
                continue
            if current_key is None: continue
            if line == "NONE":
                approvals[current_key] = []
                continue
            try:
                approvals[current_key].append(int(line))
            except ValueError: continue
    return approvals

def save_approval_database(approvals, approval_file="approved_ids.txt"):
    with open(approval_file, "w", encoding="utf-8") as f:
        for dataset_key, ids in approvals.items():
            f.write(f"[{dataset_key}]\n")
            if len(ids) == 0:
                f.write("NONE\n\n")
            else:
                for firm_id in ids:
                    f.write(f"{firm_id}\n")
                f.write("\n")

def get_dataset_key(metadata):
    return f"{metadata['country']}_{metadata['year']}"

def build_variable_dictionary(meta, variables: list) -> pd.DataFrame:
    rows = []
    variable_labels = getattr(meta, "column_names_to_labels", {})
    for var in variables:
        rows.append({"Variable": var, "Label": variable_labels.get(var, "No label available")})
    return pd.DataFrame(rows)

def build_value_labels_dictionary(meta, variables: list) -> pd.DataFrame:
    rows = []
    variables_set = {v.lower() for v in variables}
    value_labels = getattr(meta, "variable_value_labels", {})
    for variable, mapping in value_labels.items():
        if variable.lower() not in variables_set or not mapping: continue
        for value, label in mapping.items():
            rows.append({"Variable": variable, "Value": value, "Label": label})
    return pd.DataFrame(rows)

def write_excel_output(output_xlsx_path, ict_df, metadata, processing_log, dictionary_df, value_labels_df):
    with pd.ExcelWriter(output_xlsx_path, engine="openpyxl") as writer:
        ict_df.to_excel(writer, sheet_name="ICT_Firms", index=False)
        dictionary_df.to_excel(writer, sheet_name="Variable_Dictionary", index=False)
        value_labels_df.to_excel(writer, sheet_name="Value_Labels", index=False)
        pd.DataFrame(processing_log).to_excel(writer, sheet_name="Processing_Log", index=False)

    wb = load_workbook(output_xlsx_path)
    ws = wb["ICT_Firms"]
    for cell in ws[1]: cell.font = Font(bold=True)
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    for column in ws.columns:
        max_len = 0
        col_letter = get_column_letter(column[0].column)
        for cell in column:
            try: max_len = max(max_len, len(str(cell.value)))
            except: pass
        ws.column_dimensions[col_letter].width = min(max_len + 3, 50)
    wb.save(output_xlsx_path)

def screen_ict_candidates(file_path: str, keywords: list = DEFAULT_ICT_KEYWORDS, encoding: str = "utf-8") -> pd.DataFrame:
    if not os.path.isfile(file_path):
        raise FileNotFoundError(f"Dataset not found: '{file_path}'")
    log.info("Loading '%s' for ICT screening …", file_path)
    try:
        df, _ = pyreadstat.read_dta(file_path, encoding=encoding, apply_value_formats=False)
    except pyreadstat.ReadstatError:
        try:
            log.warning(f"UTF-8 failed for {file_path}, falling back to cp1252")
            df, _ = pyreadstat.read_dta(file_path, encoding="cp1252", apply_value_formats=False)
        except pyreadstat.ReadstatError:
            log.warning(f"cp1252 failed for {file_path}, falling back to latin-1")
            df, _ = pyreadstat.read_dta(file_path, encoding="latin-1", apply_value_formats=False)

    df.columns = df.columns.str.lower()
    full_isic = pd.to_numeric(df["d1a2_v4"], errors="coerce")
    df["ict_strict"] = full_isic.isin(ICT_ISIC_CLASSES)
    activity_col = next((c for c in ACTIVITY_DESC_COLS if c in df.columns), None)
    if activity_col is None:
        log.warning("No activity description column found.")
        return pd.DataFrame()
    df["matched_keywords"] = df[activity_col].apply(_match_keywords, keywords=keywords)
    df["ict_keyword_match"] = df["matched_keywords"] != ""
    candidates = df[df["ict_keyword_match"] & ~df["ict_strict"]].copy()
    print(f"\nStrict ICT: {df['ict_strict'].sum()} | Candidates: {len(candidates)}")
    if not candidates.empty:
        display(data_table.DataTable(candidates[["idstd", "d1a2_v4", activity_col, "matched_keywords"]], include_index=False, num_rows_per_page=10))
    return candidates

def extract_wbes_it_data(file_path: str, output_csv_path: str, additional_vars: Optional[list] = None, approved_extra_ict_ids: Optional[list] = None, isic_classes: list = ICT_ISIC_CLASSES, encoding: str = "utf-8") -> tuple[pd.DataFrame, "pyreadstat.metadata_container"]:
    approved_ids = approved_extra_ict_ids or []
    additional_clean = [v.lower().strip() for v in (additional_vars or []) if v and v.strip()]
    requested_vars = list(dict.fromkeys(BASE_VARS + additional_clean))

    try:
        df_full, meta = pyreadstat.read_dta(file_path, encoding=encoding, apply_value_formats=False)
    except pyreadstat.ReadstatError:
        try:
            df_full, meta = pyreadstat.read_dta(file_path, encoding="cp1252", apply_value_formats=False)
        except pyreadstat.ReadstatError:
            df_full, meta = pyreadstat.read_dta(file_path, encoding="latin-1", apply_value_formats=False)

    df_full.columns = df_full.columns.str.lower()
    meta.variable_value_labels = {k.lower(): v for k, v in meta.variable_value_labels.items()}
    metadata = parse_dataset_metadata(file_path)
    weight_var = detect_weight_variable(df_full)
    if weight_var: requested_vars.append(weight_var)
    requested_vars.extend(["country", "year", "source_file", "weight_variable_used"])
    requested_vars = list(dict.fromkeys(requested_vars))
    df_full["ict_strict"], df_full["ict_extended"] = _build_ict_masks(df_full, approved_ids, isic_classes)
    df_full["country"], df_full["year"], df_full["source_file"] = metadata["country"], metadata["year"], metadata["source_file"]
    df_full["weight_variable_used"] = weight_var if weight_var else ""
    available_cols = set(df_full.columns)
    cols_to_extract = [v for v in requested_vars if v in available_cols]
    df = df_full.loc[df_full["ict_extended"], cols_to_extract + ["ict_strict", "ict_extended"]].copy().reset_index(drop=True)
    for col in df.columns: df[col] = _replace_wbes_missing(df[col], WBES_MISSING_CODES)
    df = _apply_value_labels(df, meta, skip_cols=NUMERIC_PRESERVE)
    os.makedirs(os.path.dirname(os.path.abspath(output_csv_path)), exist_ok=True)
    return df, meta

if __name__ == "__main__":
    DATASET_FOLDER = "/content/drive/MyDrive/datasets"
    OUTPUT_FOLDER = "output"
    ENCODING = "utf-8"
    # Replaced six specific obstacle variables with the general 'm1a'
    ADDITIONAL_VARS = ["m1a", "j7a", "j30f", "j31", "h30", "e30", "j2", "j30c", "j30a", "d30b"]

    datasets = discover_datasets(DATASET_FOLDER)
    approvals = load_approval_database()

    print("\nReview Mode: Y=All, YY=Fast, ID=One")
    review_mode = input("Selection: ").strip().upper()

    selected_dataset = None
    if review_mode == "ID":
        print("\nAvailable datasets:")
        for d in datasets:
            print(get_dataset_key(parse_dataset_metadata(d)))
        selected_dataset = input("\nDataset to review: ").strip()

    # Initialize master list OUTSIDE the loop
    master_datasets = []

    for dataset in datasets:
        metadata = parse_dataset_metadata(dataset)
        dataset_key = get_dataset_key(metadata)

        # Determine if this specific dataset needs interactive review
        needs_review = (
            (review_mode == "Y") or
            (review_mode == "YY" and dataset_key not in approvals) or
            (review_mode == "ID" and dataset_key == selected_dataset)
        )

        print(f"\n{'='*70}\nProcessing {dataset_key}...\n{'='*70}")

        if needs_review:
            screen_ict_candidates(dataset, encoding=ENCODING)
            time.sleep(0.5)
            user_input = input(f"\nEnter approved IDs for {dataset_key} (comma separated, Enter for none): ").strip()
            approved_ids = [int(x.strip()) for x in user_input.split(",") if x.strip()] if user_input else []
            approvals[dataset_key] = approved_ids
            save_approval_database(approvals)
        else:
            approved_ids = approvals.get(dataset_key, [])
            print(f"Using stored approvals for {dataset_key}: {approved_ids}")

        output_file = os.path.join(OUTPUT_FOLDER, f"{dataset_key}_ICT.xlsx")
        df_final, meta = extract_wbes_it_data(dataset, output_file, ADDITIONAL_VARS, approved_ids, encoding=ENCODING)

        # Append copy to master list
        master_datasets.append(df_final.copy())

        weight_var = df_final["weight_variable_used"].iloc[0] if not df_final.empty else "None"
        proc_log = {
            "Item": ["Country", "Year", "Weight", "Strict ICT", "Final ICT"],
            "Value": [metadata['country'], metadata['year'], weight_var, int(df_final['ict_strict'].sum()), len(df_final)]
        }

        write_excel_output(
            output_file,
            df_final,
            metadata,
            proc_log,
            build_variable_dictionary(meta, list(df_final.columns)),
            build_value_labels_dictionary(meta, list(df_final.columns))
        )
        print(f"Summary: {len(df_final)} firms extracted to {output_file}")

    # BUILD MASTER DATASET AFTER ALL COUNTRIES ARE PROCESSED
    print("\n" + "=" * 70)
    print("BUILDING MASTER DATASET")
    print("=" * 70)

    if master_datasets:
        master_df = pd.concat(master_datasets, ignore_index=True)
        summary_df = master_df.groupby(["country", "year"]).size().reset_index(name="ICT_Firms")

        master_output = os.path.join(OUTPUT_FOLDER, "master_ict_dataset.xlsx")
        with pd.ExcelWriter(master_output, engine="openpyxl") as writer:
            master_df.to_excel(writer, sheet_name="Master_ICT", index=False)
            summary_df.to_excel(writer, sheet_name="Summary", index=False)

        print(f"Master dataset saved: {len(master_df)} total firms across {len(summary_df)} datasets.")
    else:
        print("No data collected for master dataset.")

In [ ]:
#This part of a code imports results back to Drive (optional)
import shutil
from google.colab import drive

# Ensure drive is mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Define paths
source_folder = 'output'
zip_filename = 'ICT_Analysis_Results'
drive_destination = '/content/drive/MyDrive/ICT_Analysis_Results.zip'

try:
    # Create a zip file from the output folder
    shutil.make_archive(zip_filename, 'zip', source_folder)

    # Move the zip file to Google Drive
    shutil.move(f'{zip_filename}.zip', drive_destination)

    print(f"Success! All output files have been zipped and saved to your Google Drive as: {drive_destination}")
except Exception as e:
    print(f"An error occurred while saving to Drive: {e}")